# Notebook 13 — Monte Carlo simulation

**Curriculum:** advanced quant track.

**Prerequisites:** work through **Notebook 01** (core types and money) and **Notebook 04** (math toolkit — statistics, stable numerics, and simulation intuition). This notebook assumes comfort with `Money`-like results and basic probability.

**In this notebook:** Monte Carlo option pricing in `finstack.monte_carlo` — analytical Black–Scholes benchmarks, European pricers, engine configuration, path-dependent and American (LSMC) examples, and a three-way comparison for an ATM European call.


## Monte Carlo option pricing

Under **risk-neutral** pricing, a European derivative value is the discounted expectation of its payoff under a model for the underlying. When no closed form exists, we **simulate** many paths of the process, average payoffs, and discount — the **Monte Carlo** estimator.

For a European call on a stock following geometric Brownian motion (GBM), the Black–Scholes formula gives the exact benchmark. Monte Carlo should agree within simulation error (stderr shrinks like \(1/\sqrt{N}\) for \(N\) paths).

`finstack.monte_carlo` exposes:

- **Analytical** `black_scholes_call` / `black_scholes_put` for sanity checks.
- **High-level pricers** such as `EuropeanPricer` and convenience `price_european_call`.
- **`McEngine`** with a **`TimeGrid`** when you want an explicit stepping setup.
- **Processes** (`GbmProcess`, `HestonProcess`, …) for model parameters and extensions.
- **Exotics:** `PathDependentPricer` (e.g. Asian), `LsmcPricer` (American via Longstaff–Schwartz).


### Black–Scholes (analytical benchmark)

Use these closed-form prices to validate Monte Carlo output and to check **put–call parity**: \(C - P = e^{-rT}(F - K)\) with forward \(F = S e^{(r-q)T}\).


In [ ]:
import math

from finstack.monte_carlo import (
    black_scholes_call,
    black_scholes_put,
    price_european_call,
    price_european_put,
    EuropeanPricer,
    McEngine,
    TimeGrid,
    GbmProcess,
    HestonProcess,
    PathDependentPricer,
    LsmcPricer,
)

spot = strike = 100.0
r = 0.05
T = 1.0
bs_call = black_scholes_call(spot, strike, r, 0.0, 0.20, T)
bs_put = black_scholes_put(spot, strike, r, 0.0, 0.20, T)
disc = math.exp(-r * T)
print(f"BS Call: {bs_call:.6f}")
print(f"BS Put: {bs_put:.6f}")
print(f"Put-Call Parity: C-P = {bs_call - bs_put:.6f}, S-K*e^(-rT) = {spot - strike * disc:.6f}")


### `EuropeanPricer` (single-step-style European MC)

`EuropeanPricer` runs a straightforward GBM simulation and returns a `MonteCarloResult`: mean estimate in `mean` (a `Money`-like amount + currency), standard error, asymptotic confidence band, and path count.


In [ ]:
pricer = EuropeanPricer(num_paths=50_000, seed=42)
result = pricer.price_call(spot=100.0, strike=100.0, rate=0.05, div_yield=0.0, vol=0.20, expiry=1.0)
print(f"MC Call: {result.mean.amount:.6f}")
print(f"Currency: {result.mean.currency.code}")
print(f"Std error: {result.stderr:.6f}")
print(f"95% CI: [{result.ci_lower.amount:.6f}, {result.ci_upper.amount:.6f}]")
print(f"Num paths: {result.num_paths}")

result_put = pricer.price_put(spot=100.0, strike=100.0, rate=0.05, div_yield=0.0, vol=0.20, expiry=1.0)
print(f"MC Put: {result_put.mean.amount:.6f}")


### `McEngine` constructor pattern

The public Python API takes the engine configuration directly in the `McEngine` constructor. Build a `TimeGrid`, then pass `num_paths`, `time_grid`, and optional controls such as `seed`, `use_parallel`, and `antithetic`.


In [ ]:
cfg_grid = TimeGrid(1.0, 252)
cfg_engine = McEngine(num_paths=50_000, time_grid=cfg_grid, seed=99)
call_result = cfg_engine.price_european_call(100.0, 100.0, 0.05, 0.0, 0.20, currency="USD")
print(f"Configured McEngine Call: {call_result.mean.amount:.6f}")


### `McEngine` with `TimeGrid`

For a lower-level pattern, construct a `TimeGrid(T, steps)`, then `McEngine(num_paths, grid, seed=...)`, and price Europeans with `price_european_call` / `price_european_put`.


In [ ]:
grid = TimeGrid(1.0, 252)
engine = McEngine(50_000, grid, seed=42)
result = engine.price_european_call(100.0, 100.0, 0.05, 0.0, 0.20)
print(f"McEngine Call: {result.mean.amount:.6f}")
result_put = engine.price_european_put(100.0, 100.0, 0.05, 0.0, 0.20)
print(f"McEngine Put: {result_put.mean.amount:.6f}")


### Stochastic processes

`GbmProcess` captures risk-neutral drift parameters for GBM. `HestonProcess` adds stochastic variance; the **`satisfies_feller`** flag reports whether the Feller condition \(2\kappa\theta > \xi^2\) holds (helps variance stay strictly positive in continuous time).


In [ ]:
gbm = GbmProcess(rate=0.05, div_yield=0.0, vol=0.20)
print(f"GBM: {gbm}")

heston = HestonProcess(
    rate=0.05, div_yield=0.0, v0=0.04,
    kappa=2.0, theta=0.04, xi=0.3, rho=-0.7
)
print(f"Heston: {heston}")
print(f"Feller satisfied: {heston.satisfies_feller}")


### Convenience functions

`price_european_call` / `price_european_put` fold path count and seed into one call — handy for scripts and quick checks.


In [ ]:
call_price = price_european_call(
    spot=100.0, strike=100.0, rate=0.05, div_yield=0.0,
    vol=0.20, expiry=1.0, num_paths=10_000, seed=42,
)
print(f"Convenience call: {call_price.mean.amount:.6f}")


### Path-dependent: arithmetic Asian call

`PathDependentPricer` prices payoffs that depend on the whole path; here an **arithmetic average** Asian call with fixings at each simulated step.


In [ ]:
asian_pricer = PathDependentPricer(num_paths=10_000, seed=42)
asian_result = asian_pricer.price_asian_call(
    spot=100.0, strike=100.0, rate=0.05, div_yield=0.0,
    vol=0.20, expiry=1.0, num_steps=252,
)
print(f"Asian Call: {asian_result.mean.amount:.6f}")


### American options: `LsmcPricer`

**Longstaff–Schwartz Monte Carlo** estimates continuation value via regression on basis functions. In continuous-time theory, an American put is **at least** as valuable as the European with the same terms; LSMC is a biased, noisy estimator, so compare against both **BS** and a **European MC** control with the same seed/paths when validating.

**Exercise grid bias:** `num_steps` fixes the Bermudan exercise grid. Too few steps **coarsens** the exercise boundary versus continuous exercise (often **biasing low** early-exercise value); using a **252-step** (roughly daily) grid is a common teaching default, though runtime grows with paths × steps.


In [ ]:
lsmc = LsmcPricer(num_paths=10_000, seed=42)
am_put = lsmc.price_american_put(
    spot=100.0, strike=100.0, rate=0.05, div_yield=0.0,
    vol=0.30, expiry=1.0, num_steps=252,
)
bs_put_ref = black_scholes_put(100.0, 100.0, 0.05, 0.0, 0.30, 1.0)
mc_euro_put = EuropeanPricer(num_paths=10_000, seed=42).price_put(
    spot=100.0, strike=100.0, rate=0.05, div_yield=0.0, vol=0.30, expiry=1.0,
)
print(f"American Put (LSMC): {am_put.mean.amount:.6f}")
print(f"European Put (BS):   {bs_put_ref:.6f}")
print(f"European Put (MC):   {mc_euro_put.mean.amount:.6f}")
print(f"LSMC >= European MC: {am_put.mean.amount >= mc_euro_put.mean.amount}")
print("(If False, treat as a signal to raise num_steps/num_paths or audit LSMC settings.)")


## Mini-example: one ATM call, three estimators

Price a European call with \(S=K=100\), \(r=5\%\), \(q=0\), \(\sigma=20\%\), \(T=1\) year using:

1. Black–Scholes analytical
2. `EuropeanPricer` with 50,000 paths
3. `McEngine` with a 252-step `TimeGrid` and 50,000 paths

The table reports each price, error versus Black–Scholes, and (for MC) the 95% confidence interval from the binding.


In [ ]:
spot = strike = 100.0
rate = 0.05
div_yield = 0.0
vol = 0.20
expiry = 1.0
num_paths = 50_000
seed = 42

bs = black_scholes_call(spot, strike, rate, div_yield, vol, expiry)

ep = EuropeanPricer(num_paths=num_paths, seed=seed)
r_ep = ep.price_call(
    spot=spot, strike=strike, rate=rate, div_yield=div_yield, vol=vol, expiry=expiry,
)

grid = TimeGrid(expiry, 252)
eng = McEngine(num_paths, grid, seed=seed)
r_me = eng.price_european_call(spot, strike, rate, div_yield, vol)

def fmt_ci(r):
    return f"[{r.ci_lower.amount:.6f}, {r.ci_upper.amount:.6f}]"

rows = [
    ("Black-Scholes (exact)", bs, 0.0, "n/a (exact)"),
    ("EuropeanPricer (50k)", r_ep.mean.amount, r_ep.mean.amount - bs, fmt_ci(r_ep)),
    ("McEngine + 252 steps (50k)", r_me.mean.amount, r_me.mean.amount - bs, fmt_ci(r_me)),
]

print(f"Parameters: S={spot}, K={strike}, r={rate}, q={div_yield}, sigma={vol}, T={expiry}y")
print()
w0, w1, w2, w3 = 28, 14, 14, 36
print(f"{'Method':<{w0}} {'Price':>{w1}} {'Err vs BS':>{w2}} {'95% CI':<{w3}}")
print("-" * (w0 + w1 + w2 + w3))
for name, px, err, ci in rows:
    print(f"{name:<{w0}} {px:>{w1}.6f} {err:>{w2}.6f} {ci:<{w3}}")


## Takeaways

- **Monte Carlo** estimates discounted risk-neutral expectations; stderr scales with \(1/\sqrt{N}\) — increase paths or use variance reduction when tight CIs matter.
- **Black–Scholes** Europeans on GBM are the standard benchmark; MC means should land inside the reported interval around the analytical value.
- **`EuropeanPricer`** and **`McEngine` + `TimeGrid`** are the main public Python entry points for GBM European pricing; choose based on how much control you want over the explicit time grid.
- **Path-dependent** and **American** payoffs need path-wise or backward schemes (`PathDependentPricer`, `LsmcPricer`); always cross-check economics (e.g. American \(\geq\) European).
- **Next steps:** combine with curves and calendars from the foundations notebooks for full term-structure setups, or explore `HestonProcess` and other processes for richer dynamics.
